In [6]:
!pip install "stable-baselines3[extra]" "gymnasium[classic_control]==0.29.1"

# 1. Импорт библиотек

In [7]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from gymnasium import spaces
from scipy.optimize import minimize
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env

# 2. Загрузка данных и калибровка весов

In [8]:
df = pd.read_excel("/content/dataset2.xlsx")
paste_columns = [
    "Т441",
    "Т449",
    "Т408",
    "Т415",
    "Т406",
    "Т423",
    "Т477",
    "Т416",
    "Т404",
    "Т403",
    "T426",
]
df[paste_columns] = df[paste_columns].fillna(0)

X = df[paste_columns].values
y_L = df["dL'"] - df["dL"]
y_A = df["dA'"] - df["dA"]
y_B = df["dB'"] - df["dB"]

W_L, _, _, _ = np.linalg.lstsq(X, y_L, rcond=None)
W_A, _, _, _ = np.linalg.lstsq(X, y_A, rcond=None)
W_B, _, _, _ = np.linalg.lstsq(X, y_B, rcond=None)

N_PASTES = 11
paste_names = [
    "Т441",
    "Т449",
    "Т408",
    "Т415",
    "Т406",
    "Т423",
    "Т477",
    "Т416",
    "Т404",
    "Т403",
    "T426",
]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# 3. Симулятор среды

In [9]:
def apply_pastes_nonlinear(dL0, dA0, dB0, p):
    assert len(p) == 11
    p_total = np.sum(p)

    delta_L_lin = np.dot(W_L, p)
    delta_A_lin = np.dot(W_A, p)
    delta_B_lin = np.dot(W_B, p)

    nonlinear_scale = 1 + 1.2 * (1 - np.exp(-p_total * 2))
    saturation = 1 - 0.3 * np.tanh(p_total * 3)

    cross_L = -0.5 * p[0] * p[1] if p[0] > 0 and p[1] > 0 else 0
    cross_A = -1.2 * p[0] ** 1.5 if p[0] > 0 else 0

    dL1 = dL0 + (delta_L_lin + cross_L) * nonlinear_scale * saturation
    dA1 = dA0 + (delta_A_lin + cross_A) * nonlinear_scale * saturation
    dB1 = dB0 + delta_B_lin * nonlinear_scale * saturation
    dE1 = np.sqrt(dL1**2 + dA1**2 + dB1**2)

    return dL1, dA1, dB1, dE1

# 4. Среда Gymnasium

In [10]:
class ColorEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=60, step_size=0.05):
        super().__init__()
        self.paste_names = paste_names
        self.N = N_PASTES
        self.step_size = step_size
        self.max_steps = max_steps

        low_obs = np.array([-10, -10, -10, 0] + [0.0] * self.N, dtype=np.float32)
        high_obs = np.array([10, 10, 10, 20] + [3.0] * self.N, dtype=np.float32)
        self.observation_space = spaces.Box(
            low=low_obs, high=high_obs, dtype=np.float32
        )
        self.action_space = spaces.Discrete(self.N + 1)

        self.tol_L, self.tol_A, self.tol_B, self.tol_E = 2.0, 0.5, 0.5, 2.0
        self.state = None
        self.steps = 0
        self.initial_color = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        dL0 = np.random.uniform(-4, 4)
        dA0 = np.random.uniform(-1, 1)
        dB0 = np.random.uniform(-1, 1)
        dE0 = np.sqrt(dL0**2 + dA0**2 + dB0**2)

        self.initial_color = (dL0, dA0, dB0)
        p0 = np.zeros(self.N, dtype=np.float32)
        self.state = np.array([dL0, dA0, dB0, dE0] + list(p0), dtype=np.float32)
        self.steps = 0
        return self.state, {}

    def step(self, action):
        dL0, dA0, dB0 = self.initial_color
        p = self.state[4:].copy()
        self.steps += 1

        done = False
        stopped_by_agent = False
        dL_curr, dA_curr, dB_curr, dE_curr = self.state[:4]

        if action == self.N:
            done = True
            stopped_by_agent = True
        else:
            p[action] += self.step_size
            p_sum = p.sum()
            used_pastes = np.count_nonzero(p > 0.01)

            if p_sum > 3.0 or used_pastes > 3 or self.steps >= self.max_steps:
                done = True
            else:
                dL, dA, dB, dE = apply_pastes_nonlinear(dL0, dA0, dB0, p)
                self.state = np.array([dL, dA, dB, dE] + list(p), dtype=np.float32)
                return self.state, -0.05, False, False, {}

        self.state = np.array(
            [dL_curr, dA_curr, dB_curr, dE_curr] + list(p), dtype=np.float32
        )
        reward = self._compute_reward(stopped_by_agent, done)
        return self.state, reward, True, False, {}

    def _compute_reward(self, stopped_by_agent, done):
        dL, dA, dB, dE = self.state[:4]
        p = self.state[4:]

        err_L = abs(dL) / 2.0
        err_A = 4.0 * (abs(dA) / 0.5)
        err_B = 4.0 * (abs(dB) / 0.5)

        quality_reward = 20.0 - 10.0 * (err_L + err_A + err_B)
        p_sum = p.sum()
        used_pastes = np.count_nonzero(p > 0.01)
        paste_penalty = -0.05 * p_sum - 0.2 * max(0, used_pastes - 3)
        bonus_stop = 10.0 if stopped_by_agent else 0.0

        return quality_reward + paste_penalty + bonus_stop


env = ColorEnv()
check_env(env)

# 5. Конфигурация агента

In [18]:
from stable_baselines3.common.buffers import PrioritizedReplayBuffer

policy_kwargs = dict(net_arch=[64, 64])

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=5e-4,
    buffer_size=10000,
    buffer_class=PrioritizedReplayBuffer,
    learning_starts=1000,
    batch_size=32,
    gamma=0.95,
    train_freq=2,
    exploration_fraction=0.15,
    policy_kwargs=policy_kwargs,
    verbose=0,
)

model.learn(total_timesteps=30000, progress_bar=True)
model.save("color_dqn_fast")

Output()

# 6. Оптимизатор L-BFGS-B

In [20]:
def production_optimizer_11(dL0, dA0, dB0, max_total=2.8):
    def cost(p):
        dL1, dA1, dB1, _ = apply_pastes_nonlinear(dL0, dA0, dB0, p)
        err = abs(dL1) / 2.0 + 4 * abs(dA1) / 0.5 + 4 * abs(dB1) / 0.5
        penalty = 10 * max(0, p.sum() - max_total) + 20 * max(0, sum(p > 0.05) - 3)
        return err + penalty

    p_init = np.zeros(11)
    if abs(dL0) > 0.5: p_init[0] = min(0.8, abs(dL0) * 0.25)
    if abs(dA0) > 0.3: p_init[1] = min(0.8, abs(dA0) * 0.2)
    if abs(dB0) > 0.3: p_init[2] = min(0.8, abs(dB0) * 0.15)

    bounds = []
    for i in range(11):
        if p_init[i] > 0:
            bounds.append((0, 1.0))
        else:
            bounds.append((0, 0.0))

    result = minimize(
        cost, p_init, bounds=bounds, method="L-BFGS-B", options={"maxiter": 20}
    )

    p_opt = np.clip(result.x, 0, 1.0)
    dL1, dA1, dB1, dE1 = apply_pastes_nonlinear(dL0, dA0, dB0, p_opt)
    success = abs(dL1) <= 2.0 and abs(dA1) <= 0.5 and abs(dB1) <= 0.5
    recipe = {
        paste_names[i]: round(p_opt[i], 3)
        for i in range(11)
        if p_opt[i] > 0.05
    }

    return {
        "recipe": recipe,
        "predicted": (round(dL1, 2), round(dA1, 2), round(dB1, 2), round(dE1, 2)),
        "success": success,
        "total_paste": round(sum(p_opt), 2),
    }

# 7. Рассчет рецептуры

In [21]:
def get_production_recipe(dL, dA, dB):
    recipe_id = np.random.randint(1000, 9999)
    obs = np.array(
        [dL, dA, dB, np.sqrt(dL**2 + dA**2 + dB**2)] + [0.0] * 11,
        dtype=np.float32,
    )
    action, _ = model.predict(obs, deterministic=True)

    result = production_optimizer_11(dL, dA, dB)
    grams_total = result["total_paste"] * 200
    status = "Конечные" if result["success"] else "Контроль"

    dL1, dA1, dB1, dE1 = result["predicted"]

    print(f"\nРецепт #{recipe_id:04d}")
    print(f"Стартовые  dL = {dL:.1f}; dA = {dA:.1f}; dB = {dB:.1f}")
    print(f"{status} dL = {dL1:+.1f};  dA = {dA1:+.1f}; dB = {dB1:+.1f}")
    print(f"dE = {dE1:.2f}")
    print(f"Добавлено {result['total_paste']:.1f}% паст")

    if result["recipe"]:
        print(f"на 20 кг краски ({grams_total:.0f}г всего):")
        for paste, pct in result["recipe"].items():
            grams = pct * 200
            print(f"   {paste}: {grams:.0f}г")
    else:
        print("Пасты не требуются")

# Испытание на случайной пробе из цеха

In [22]:
get_production_recipe(3.21, 3.75, -8.57);


Рецепт #4368
Стартовые  dL = 3.2; dA = 3.8; dB = -8.6
Конечные dL = +1.8;  dA = -0.4; dB = -0.0
dE = 1.79
Добавлено 2.2% паст
на 20 кг краски (432г всего):
   Т441: 61г
   Т449: 54г
   Т408: 60г
   Т404: 121г
   Т403: 118г
